In [1]:
from anthropic.types import ToolParam
from datetime import datetime
import json
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [2]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam({
  "name": "get_current_datetime",
  "description": "Returns the current local date and time as a string, formatted with a Python strftime pattern. Uses the host system's local timezone; no timezone conversion is performed. Call this when the user asks for the current date or time, or when a calculation depends on the present moment.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A Python strftime format string that controls the output. Common directives: %Y (4-digit year), %m (month 01-12), %d (day 01-31), %H (hour 00-23), %M (minute), %S (second), %A (weekday name), %B (month name), %p (AM/PM). Literal text is copied as-is. Omit to use the default.",
        "default": "%Y-%m-%d %H:%M:%S",
        "minLength": 1,
        "examples": [
          "%Y-%m-%d %H:%M:%S",
          "%Y-%m-%d",
          "%H:%M",
          "%A, %B %d, %Y"
        ]
      }
    },
    "required": []
  }
})

In [3]:
# Default format: "2024-01-15 14:30:25"
#get_current_datetime()

# Just hour and minute: "14:30"
#get_current_datetime("%H:%M")

In [4]:
messages = []
messages.append({
    "role": "user",
    "content": "What is the exact time, formatted as HH:MM:SS?"
})

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

In [5]:
print(response)

Message(id='msg_011CfxRJGs6NhoDzHsoeeRgU', container=None, content=[ToolUseBlock(id='toolu_01FWbMGeomJxvhndv1KEjw8P', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)], diagnostics=None, model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=787, output_tokens=63, output_tokens_details=None, server_tool_use=None, service_tier='standard'))


In [6]:
messages.append({
    "role": "assistant",
    "content": response.content
})

In [8]:
tool_result = get_current_datetime(**response.content[0].input)

In [9]:
print(tool_result)

08:31:31


In [11]:
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": response.content[0].id,
        "content": tool_result,
        "is_error": False
    }]
})

In [12]:
print(messages)

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01FWbMGeomJxvhndv1KEjw8P', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_01FWbMGeomJxvhndv1KEjw8P', 'content': '08:31:31', 'is_error': False}]}]


In [13]:
response = client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = messages,
    tools = [get_current_datetime_schema]
)

In [14]:
print(response)

Message(id='msg_011CfxSJWDdSCctzVWPH8TSz', container=None, content=[TextBlock(citations=None, text='The exact time is **08:31:31** (8 hours, 31 minutes, and 31 seconds).', type='text')], diagnostics=None, model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=867, output_tokens=30, output_tokens_details=None, server_tool_use=None, service_tier='standard'))
